In [ ]:
import pandas as pd
import numpy as np
import os

# Update BASE_DIR to match your local installation if needed
BASE_DIR = r"C:\Users\Admin\Final_review"
csv_path = os.path.join(BASE_DIR, "datasets", "FTSE.csv")

df = pd.read_csv(csv_path)
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"Dataset : FTSE 100")
print(f"Shape   : {df.shape}")
print(f"Date range: {df.index[0].date()} -> {df.index[-1].date()}")
print(df.dtypes)
df.head()

# ADF on raw prices

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(df['price'], autolag='AIC')

print("=== Augmented Dickey-Fuller Test on Price ===")
print(f"Test Statistic   : {result[0]:.4f}")
print(f"p-value          : {result[1]:.4f}")
print(f"Lags Used        : {result[2]}")
print(f"Observations     : {result[3]}")
print("\nCritical Values:")
for key, val in result[4].items():
    print(f"   {key} : {val:.4f}")

print("\n--- Interpretation ---")
if result[1] > 0.05:
    print("Fail to reject H0 -> Series is NON-STATIONARY (expected for price levels)")
else:
    print("Reject H0 -> Series is STATIONARY")

# Log return transformation applied

In [ ]:
df['log_returns'] = np.log(df['price'] / df['price'].shift(1))
df.dropna(subset=['log_returns'], inplace=True)

print("=== Log Returns Summary Statistics ===")
print(df['log_returns'].describe())
print(f"\nSkewness : {df['log_returns'].skew():.4f}")
print(f"Kurtosis : {df['log_returns'].kurtosis():.4f}  (excess — fat tails if > 0)")
print(f"Total rows after transformation : {len(df)}")

# ADF test on log_returns

In [ ]:
result = adfuller(df['log_returns'], autolag='AIC')

print("=== Augmented Dickey-Fuller Test on Log Returns ===")
print(f"Test Statistic   : {result[0]:.4f}")
print(f"p-value          : {result[1]:.4f}")
print(f"Lags Used        : {result[2]}")
print(f"Observations     : {result[3]}")
print("\nCritical Values:")
for key, val in result[4].items():
    print(f"   {key} : {val:.4f}")

print("\n--- Interpretation ---")
if result[1] < 0.05:
    print("Reject H0 -> Log Returns are STATIONARY -- safe to proceed")
else:
    print("Fail to reject H0 -> Non-stationary -- investigate further")

# Outlier detection

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats

# Z-Score method
df['z_score'] = (df['log_returns'] - df['log_returns'].mean()) / df['log_returns'].std()
z_outliers = df[np.abs(df['z_score']) > 3]

print("=== Z-Score Outlier Detection (threshold = +-3) ===")
print(f"Total Outliers Detected : {len(z_outliers)}")
print(f"As % of dataset         : {len(z_outliers)/len(df)*100:.2f}%")
print("\nTop 10 Extreme Outliers:")
print(z_outliers['log_returns'].sort_values(key=abs, ascending=False).head(10))

# IQR method
Q1 = df['log_returns'].quantile(0.25)
Q3 = df['log_returns'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR
iqr_outliers = df[(df['log_returns'] < lower_bound) | (df['log_returns'] > upper_bound)]
print(f"\n=== IQR Outlier Detection (3xIQR) ===")
print(f"Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]  Outliers: {len(iqr_outliers)}")

# Plots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Outlier Detection -- FTSE 100 Log Returns', fontsize=14, fontweight='bold')

axes[0,0].plot(df.index, df['log_returns'], color='steelblue', linewidth=0.7, label='Log Returns')
axes[0,0].scatter(z_outliers.index, z_outliers['log_returns'], color='red', s=15, zorder=5, label='Z-Score Outliers')
axes[0,0].axhline(0, color='black', linewidth=0.5)
axes[0,0].set_title('Log Returns with Z-Score Outliers')
axes[0,0].legend()

axes[0,1].hist(df['log_returns'], bins=100, color='steelblue', edgecolor='white', alpha=0.7)
axes[0,1].axvline(lower_bound, color='red',   linestyle='--', linewidth=1.5, label=f'IQR Lower: {lower_bound:.4f}')
axes[0,1].axvline(upper_bound, color='green', linestyle='--', linewidth=1.5, label=f'IQR Upper: {upper_bound:.4f}')
axes[0,1].set_title('Return Distribution with IQR Bounds')
axes[0,1].legend()

axes[1,0].boxplot(df['log_returns'].dropna(), vert=True, patch_artist=True,
                  boxprops=dict(facecolor='steelblue', color='white'),
                  medianprops=dict(color='yellow', linewidth=2),
                  flierprops=dict(marker='o', color='red', markersize=3))
axes[1,0].set_title('Boxplot of Log Returns')

stats.probplot(df['log_returns'].dropna(), dist="norm", plot=axes[1,1])
axes[1,1].set_title('QQ Plot -- Fat Tail Check')
axes[1,1].get_lines()[0].set(color='steelblue', markersize=2)
axes[1,1].get_lines()[1].set(color='red', linewidth=1.5)

plt.tight_layout()
plt.show()

# Volatility clustering

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(3, 1, figsize=(16, 14))
fig.suptitle('Volatility Clustering Analysis -- FTSE 100 Log Returns', fontsize=14, fontweight='bold')

axes[0].plot(df.index, df['log_returns'], color='steelblue', linewidth=0.7)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('Daily Log Returns')
axes[0].set_ylabel('Log Return')

df['rolling_vol_21'] = df['log_returns'].rolling(window=21).std() * np.sqrt(252)
axes[1].plot(df.index, df['rolling_vol_21'], color='darkorange', linewidth=0.8)
axes[1].axhline(df['rolling_vol_21'].mean(), color='red', linestyle='--',
                linewidth=1.2, label=f"Mean Vol: {df['rolling_vol_21'].mean():.4f}")
axes[1].set_title('Rolling 21-Day Annualised Volatility')
axes[1].legend()

df['squared_returns'] = df['log_returns'] ** 2
axes[2].plot(df.index, df['squared_returns'], color='purple', linewidth=0.6)
axes[2].set_title('Squared Log Returns -- Variance Proxy')

plt.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(14, 4))
plot_acf(df['squared_returns'].dropna(), lags=50, ax=ax2, color='purple',
         title='ACF of Squared Returns -- Volatility Clustering Confirmation')
plt.tight_layout()
plt.show()

# Fitting GJR-GARCH(1,1) with Student-t Residuals

In [ ]:
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

returns_scaled = df['log_returns'] * 100

gjr_garch = arch_model(
    returns_scaled,
    vol='Garch', p=1, o=1, q=1,   # o=1 makes it GJR-GARCH (asymmetry/leverage term)
    dist='t'                        # Student-t residuals
)
gjr_result = gjr_garch.fit(disp='off')

print(gjr_result.summary())

# Store conditional volatility (rescaled to daily)
df['garch_vol'] = gjr_result.conditional_volatility / 100

print("\n=== Conditional Volatility Sanity Check ===")
print(df['garch_vol'].describe())

params = gjr_result.params
persistence = params['alpha[1]'] + params['beta[1]'] + 0.5 * params['gamma[1]']
print("\n=== Key GJR-GARCH(1,1) Parameters ===")
print(f"Omega (baseline variance)  : {params['omega']:.6f}")
print(f"Alpha (ARCH shock impact)  : {params['alpha[1]']:.4f}")
print(f"Gamma (leverage effect)    : {params['gamma[1]']:.4f}  ({'leverage confirmed' if params['gamma[1]'] > 0 else 'not significant'})")
print(f"Beta  (vol persistence)    : {params['beta[1]']:.4f}")
print(f"Nu    (Student-t d.o.f.)   : {params['nu']:.4f}")
print(f"Persistence (a+b+0.5g)     : {persistence:.4f}")

# Visualisation
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('GJR-GARCH(1,1) -- Conditional Volatility -- FTSE 100', fontsize=14, fontweight='bold')

axes[0].plot(df.index, df['garch_vol'], color='darkorange', linewidth=0.8, label='GARCH Cond. Vol')
axes[0].plot(df.index, df['rolling_vol_21']/np.sqrt(252), color='steelblue', linewidth=0.8,
             alpha=0.6, label='Rolling 21-Day Vol (daily scale)')
axes[0].axhline(df['garch_vol'].mean(), color='red', linestyle='--', linewidth=1.2,
                label=f"Mean: {df['garch_vol'].mean():.4f}")
axes[0].set_title('GARCH Conditional Volatility vs Rolling Volatility')
axes[0].legend()

axes[1].plot(df.index, df['log_returns'], color='steelblue', linewidth=0.6, alpha=0.7, label='Log Returns')
axes[1].plot(df.index,  2*df['garch_vol'], color='red', linewidth=0.8, linestyle='--', label='+2sigma GARCH Band')
axes[1].plot(df.index, -2*df['garch_vol'], color='red', linewidth=0.8, linestyle='--', label='-2sigma GARCH Band')
axes[1].set_title('Log Returns with +-2sigma GARCH Bands')
axes[1].legend()

plt.tight_layout()
plt.show()

# Building the LSTM feature matrix

In [ ]:
from sklearn.preprocessing import MinMaxScaler

df['abs_returns']     = np.abs(df['log_returns'])
df['squared_returns'] = df['log_returns'] ** 2

features = ['log_returns', 'garch_vol', 'rolling_vol_21', 'squared_returns', 'abs_returns']

# Target: next-day realised volatility (absolute log return)
df['target'] = np.abs(df['log_returns'].shift(-1))
df.dropna(subset=['target'] + features, inplace=True)

# Chronological 70/15/15 split
train_size = int(len(df) * 0.70)
val_size   = int(len(df) * 0.15)

train_df = df.iloc[:train_size]
val_df   = df.iloc[train_size:train_size + val_size]
test_df  = df.iloc[train_size + val_size:]

print("=== Chronological Train/Val/Test Split ===")
print(f"Train : {train_df.index[0].date()} -> {train_df.index[-1].date()}  ({len(train_df)} rows -- 70%)")
print(f"Val   : {val_df.index[0].date()}   -> {val_df.index[-1].date()}  ({len(val_df)} rows -- 15%)")
print(f"Test  : {test_df.index[0].date()}  -> {test_df.index[-1].date()}  ({len(test_df)} rows -- 15%)")

# Scale -- fit on train only (no data leakage)
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

X_train_scaled = scaler_X.fit_transform(train_df[features])
X_val_scaled   = scaler_X.transform(val_df[features])
X_test_scaled  = scaler_X.transform(test_df[features])

y_train_scaled = scaler_y.fit_transform(train_df[['target']])
y_val_scaled   = scaler_y.transform(val_df[['target']])
y_test_scaled  = scaler_y.transform(test_df[['target']])

# Build 21-day rolling sequences
SEQUENCE_LENGTH = 21

def build_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = build_sequences(X_train_scaled, y_train_scaled, SEQUENCE_LENGTH)
X_val_seq,   y_val_seq   = build_sequences(X_val_scaled,   y_val_scaled,   SEQUENCE_LENGTH)
X_test_seq,  y_test_seq  = build_sequences(X_test_scaled,  y_test_scaled,  SEQUENCE_LENGTH)

print("\n=== Sequence Shapes (samples, timesteps, features) ===")
print(f"X_train_seq : {X_train_seq.shape}")
print(f"X_val_seq   : {X_val_seq.shape}")
print(f"X_test_seq  : {X_test_seq.shape}")

# LSTM Training

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)
tf.keras.backend.clear_session()

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, len(features))),
    BatchNormalization(),
    Dropout(0.2),

    LSTM(32, return_sequences=False),
    BatchNormalization(),
    Dropout(0.2),

    Dense(16, activation='relu'),
    Dropout(0.1),
    Dense(1, activation='linear')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='huber', metrics=['mae'])
print("=== Model Architecture -- FTSE 100 ===")
model.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr      = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)

print("\n=== Training Started ===")
history = model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=100, batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

test_loss, test_mae = model.evaluate(X_test_seq, y_test_seq, verbose=0)
print(f"\n=== Test Set Performance ===")
print(f"Test Loss (Huber) : {test_loss:.6f}")
print(f"Test MAE          : {test_mae:.6f}")

y_pred_scaled = model.predict(X_test_seq)
y_pred        = scaler_y.inverse_transform(y_pred_scaled)
y_actual      = scaler_y.inverse_transform(y_test_seq)

In [ ]:
# Enhanced Training Diagnostics

train_loss = history.history['loss']
val_loss   = history.history['val_loss']
train_mae  = history.history['mae']
val_mae    = history.history['val_mae']
epochs_ran = range(1, len(train_loss) + 1)

best_epoch    = np.argmin(val_loss) + 1
best_val_loss = np.min(val_loss)
stopped_epoch = len(train_loss)

# Detect LR reduction epochs (every 7 epochs without improvement)
lr_reduction_epochs = []
patience_counter, min_seen = 0, val_loss[0]
for i in range(1, len(val_loss)):
    if val_loss[i] < min_seen:
        min_seen = val_loss[i]; patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter == 7:
            lr_reduction_epochs.append(i + 1); patience_counter = 0

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('LSTM Training Diagnostics -- FTSE 100', fontsize=14, fontweight='bold')

for ax, train_m, val_m, mname in zip(
        axes,
        [train_loss, train_mae],
        [val_loss,   val_mae],
        ['Huber Loss', 'MAE']):
    ax.plot(epochs_ran, train_m, color='steelblue',  linewidth=1.2, label=f'Train {mname}')
    ax.plot(epochs_ran, val_m,   color='darkorange', linewidth=1.2, label=f'Val {mname}')
    bval = val_m[best_epoch - 1]
    ax.axvline(best_epoch, color='green', linestyle='--', linewidth=1.2, label=f'Best Epoch: {best_epoch}')
    ax.scatter(best_epoch, bval, color='green', s=80, zorder=5)
    ax.annotate(f'  Best\n  E{best_epoch}\n  {bval:.5f}',
                xy=(best_epoch, bval),
                xytext=(best_epoch + len(epochs_ran)*0.05, bval),
                fontsize=8, color='green')
    ax.axvline(stopped_epoch, color='red', linestyle=':', linewidth=1.2,
               label=f'Early Stop: E{stopped_epoch}')
    for idx, ep in enumerate(lr_reduction_epochs):
        ax.axvline(ep, color='purple', linestyle='-.', linewidth=0.8, alpha=0.7,
                   label='LR Reduced' if idx == 0 else '')
        ax.annotate('LR down', xy=(ep, max(val_m)*0.95), fontsize=7, color='purple', ha='center')
    ax.set_xlabel('Epoch'); ax.set_ylabel(mname); ax.set_title(f'{mname} over Epochs')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=== Training Summary ===")
print(f"Best Epoch     : {best_epoch}  |  Best Val Loss : {best_val_loss:.6f}")
print(f"Stopped Epoch  : {stopped_epoch}  |  Overfit Gap   : {val_loss[-1]-train_loss[-1]:.6f}")
print(f"LR Reductions  : {lr_reduction_epochs if lr_reduction_epochs else 'None'}")

# Predicted vs Actual Volatility
test_idx = test_df.index[SEQUENCE_LENGTH:]
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(test_idx, y_actual, color='steelblue',  linewidth=0.8, label='Actual Realised Vol')
ax.plot(test_idx, y_pred,   color='darkorange', linewidth=0.8, alpha=0.8, label='LSTM Predicted Vol')
ax.set_title('LSTM -- Predicted vs Actual Realised Volatility -- FTSE 100', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Abs Log Return')
ax.legend(); plt.tight_layout(); plt.show()

# Ensemble VaR and ES Computation

In [ ]:
from scipy import stats

# Align GARCH vol and LSTM predictions on the test period
test_index     = test_df.index[SEQUENCE_LENGTH:]
garch_vol_test = df.loc[test_index, 'garch_vol'].values
lstm_vol_test  = y_pred.flatten()
actual_returns = df.loc[test_index, 'log_returns'].values

print("=== Alignment Check ===")
print(f"Test observations: {len(test_index)}")
assert len(garch_vol_test) == len(lstm_vol_test) == len(actual_returns), "Alignment error!"
print("Alignment OK")

# Weighted ensemble (60% GARCH / 40% LSTM)
GARCH_WEIGHT = 0.6
LSTM_WEIGHT  = 0.4
ensemble_vol = GARCH_WEIGHT * garch_vol_test + LSTM_WEIGHT * lstm_vol_test

print(f"\n=== Ensemble Volatility ===")
print(f"GARCH   Mean: {garch_vol_test.mean():.4f}  Std: {garch_vol_test.std():.4f}")
print(f"LSTM    Mean: {lstm_vol_test.mean():.4f}  Std: {lstm_vol_test.std():.4f}")
print(f"Ensemble Mean: {ensemble_vol.mean():.4f}  Std: {ensemble_vol.std():.4f}")

# Student-t VaR and ES (nu from GJR-GARCH fit)
nu = gjr_result.params['nu']
confidence_levels = [0.95, 0.99]
var_results = {}
es_results  = {}

for cl in confidence_levels:
    alpha  = 1 - cl
    t_q    = stats.t.ppf(alpha, df=nu)
    VaR    = -ensemble_vol * t_q
    t_pdf  = stats.t.pdf(t_q, df=nu)
    ES     = ensemble_vol * (t_pdf / alpha) * ((nu + t_q**2) / (nu - 1))
    var_results[cl] = VaR
    es_results[cl]  = ES
    breaches = actual_returns < -VaR
    print(f"\n{int(cl*100)}% -- Mean VaR: {VaR.mean()*100:.2f}%  Breach Rate: {breaches.mean()*100:.2f}%  (Expected: {alpha*100:.1f}%)")

# Visualisation
fig, axes = plt.subplots(3, 1, figsize=(16, 15))
fig.suptitle('Ensemble GARCH+LSTM -- VaR & ES -- FTSE 100', fontsize=14, fontweight='bold')

axes[0].plot(test_index, garch_vol_test, color='darkorange', linewidth=0.8, alpha=0.8, label='GARCH Vol')
axes[0].plot(test_index, lstm_vol_test,  color='steelblue',  linewidth=0.8, alpha=0.8, label='LSTM Vol')
axes[0].plot(test_index, ensemble_vol,   color='green',      linewidth=1.2, label='Ensemble Vol')
axes[0].set_title('Volatility Components'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(test_index, actual_returns, color='steelblue', linewidth=0.7, alpha=0.8, label='Returns')
axes[1].plot(test_index, -var_results[0.95], color='orange', linewidth=1.2, linestyle='--', label='95% VaR')
axes[1].plot(test_index, -var_results[0.99], color='red',    linewidth=1.2, linestyle='--', label='99% VaR')
b95 = actual_returns < -var_results[0.95]
b99 = actual_returns < -var_results[0.99]
axes[1].scatter(test_index[b95], actual_returns[b95], color='orange', s=20, zorder=5, label=f'95% Breaches ({b95.sum()})')
axes[1].scatter(test_index[b99], actual_returns[b99], color='red',    s=25, zorder=6, label=f'99% Breaches ({b99.sum()})')
axes[1].set_title('Returns vs VaR Thresholds'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].fill_between(test_index, -var_results[0.99], -es_results[0.99], alpha=0.3, color='red',    label='ES Zone 99%')
axes[2].fill_between(test_index, -var_results[0.95], -var_results[0.99],alpha=0.3, color='orange', label='VaR Zone 95-99%')
axes[2].plot(test_index, -var_results[0.95], color='orange',  linewidth=1.0, linestyle='--', label='95% VaR')
axes[2].plot(test_index, -var_results[0.99], color='red',     linewidth=1.0, linestyle='--', label='99% VaR')
axes[2].plot(test_index, -es_results[0.99],  color='darkred', linewidth=1.2, label='99% ES')
axes[2].plot(test_index, actual_returns,     color='steelblue',linewidth=0.6, alpha=0.5, label='Returns')
axes[2].set_title('Dynamic VaR & ES Risk Zones'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Volatility Scaling Buffer and Recalibrated VaR/ES

In [ ]:
# Grid search: find buffer k minimising |actual 99% breach rate - 1%|
print("=== Buffer Multiplier Grid Search ===")
print(f"{'Buffer':>8} {'95% Breach':>12} {'99% Breach':>12} {'95% Status':>14} {'99% Status':>14}")
print("-" * 65)

buffer_range = np.arange(1.0, 1.51, 0.05)
best_buffer  = 1.0
best_99_gap  = np.inf

for buf in buffer_range:
    scaled_vol = ensemble_vol * buf
    res = {}
    for cl in [0.95, 0.99]:
        alpha = 1 - cl
        t_q   = stats.t.ppf(alpha, df=nu)
        VaR   = -scaled_vol * t_q
        res[cl] = (actual_returns < -VaR).mean() * 100
    gap = abs(res[0.99] - 1.00)
    if gap < best_99_gap:
        best_99_gap = gap; best_buffer = buf
    s95 = "OK" if res[0.95] <= 6.0  else "High"
    s99 = "OK" if res[0.99] <= 1.20 else "High"
    print(f"{buf:>8.2f} {res[0.95]:>11.2f}% {res[0.99]:>11.2f}%  {s95:>14} {s99:>14}")

print(f"\nOptimal Buffer k = {best_buffer:.2f}")

ensemble_vol_buffered = ensemble_vol * best_buffer

var_buffered   = {}
es_buffered    = {}
breach_buffered = {}

for cl in confidence_levels:
    alpha = 1 - cl
    t_q   = stats.t.ppf(alpha, df=nu)
    VaR   = -ensemble_vol_buffered * t_q
    t_pdf = stats.t.pdf(t_q, df=nu)
    ES    = ensemble_vol_buffered * (t_pdf / alpha) * ((nu + t_q**2) / (nu - 1))
    var_buffered[cl]    = VaR
    es_buffered[cl]     = ES
    breach_buffered[cl] = actual_returns < -VaR
    rate = breach_buffered[cl].mean()
    print(f"\n{int(cl*100)}% Buffered -- Breach Rate: {rate*100:.2f}%  (Expected: {(1-cl)*100:.1f}%)")

# Summary table
print("\n" + "="*62)
print("   RECALIBRATED ENSEMBLE -- FINAL RISK SUMMARY")
print("="*62)
print(f"  Buffer k={best_buffer:.2f}  |  GARCH:{GARCH_WEIGHT}  LSTM:{LSTM_WEIGHT}  |  Student-t nu={nu:.2f}")
print("="*62)
print(f"{'Metric':<35} {'95% CL':>10} {'99% CL':>10}")
print("-"*62)
for label, v95, v99 in [
    ("Mean Daily VaR",  var_buffered[0.95].mean(),  var_buffered[0.99].mean()),
    ("Mean Daily ES",   es_buffered[0.95].mean(),   es_buffered[0.99].mean()),
    ("Max  Daily VaR",  var_buffered[0.95].max(),   var_buffered[0.99].max()),
    ("VaR Breach Rate", breach_buffered[0.95].mean(), breach_buffered[0.99].mean()),
]:
    print(f"{label:<35} {v95*100:>9.2f}%  {v99*100:>9.2f}%")
print("="*62)

# Final Model Summary and Risk Report

In [ ]:
from datetime import datetime
import pandas as pd

report_date = datetime.now().strftime("%B %d, %Y")
test_start  = test_df.index[SEQUENCE_LENGTH].strftime("%Y-%m-%d")
test_end    = test_df.index[-1].strftime("%Y-%m-%d")
persistence = gjr_result.params['alpha[1]'] + gjr_result.params['beta[1]'] + 0.5*gjr_result.params['gamma[1]']

annual_var_95 = var_buffered[0.95].mean() * np.sqrt(252)
annual_var_99 = var_buffered[0.99].mean() * np.sqrt(252)
annual_es_95  = es_buffered[0.95].mean()  * np.sqrt(252)
annual_es_99  = es_buffered[0.99].mean()  * np.sqrt(252)

# 10-panel comprehensive risk report
fig = plt.figure(figsize=(20, 24))
fig.suptitle('Ensemble GARCH+LSTM -- Complete Risk Report\nFTSE 100 Daily Returns',
             fontsize=16, fontweight='bold', y=0.98)

ax1 = fig.add_subplot(5, 2, (1, 2))
ax1.plot(df.index, df['log_returns'], color='steelblue', linewidth=0.6, alpha=0.8)
ax1.axvline(pd.Timestamp(train_df.index[-1]), color='orange', linestyle='--', linewidth=1.2, label='Train/Val Split')
ax1.axvline(pd.Timestamp(test_start),         color='red',    linestyle='--', linewidth=1.2, label='Val/Test Split')
ax1.set_title('Full FTSE 100 Log Return Series with Train/Val/Test Splits', fontweight='bold')
ax1.set_ylabel('Log Return'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(5, 2, (3, 4))
ax2.plot(df.index, df['garch_vol'], color='darkorange', linewidth=0.7)
ax2.axhline(df['garch_vol'].mean(), color='red', linestyle='--', linewidth=1.0,
            label=f"Mean: {df['garch_vol'].mean():.4f}")
ax2.set_title('GJR-GARCH(1,1) Conditional Volatility -- Full Period', fontweight='bold')
ax2.set_ylabel('Daily Volatility'); ax2.legend(); ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(5, 2, 5)
ax3.plot(test_index, y_actual, color='steelblue',  linewidth=0.8, label='Actual Realised Vol')
ax3.plot(test_index, y_pred,   color='darkorange', linewidth=0.8, alpha=0.8, label='LSTM Predicted Vol')
ax3.set_title('LSTM -- Predicted vs Actual Realised Volatility', fontweight='bold')
ax3.legend(fontsize=7); ax3.grid(True, alpha=0.3)

ax4 = fig.add_subplot(5, 2, 6)
ax4.plot(test_index, ensemble_vol_buffered, color='green',      linewidth=0.9, label='Buffered Ensemble')
ax4.plot(test_index, garch_vol_test,        color='darkorange', linewidth=0.7, alpha=0.6, linestyle='--', label='GARCH')
ax4.plot(test_index, lstm_vol_test,         color='steelblue',  linewidth=0.7, alpha=0.6, linestyle='--', label='LSTM')
ax4.set_title('Ensemble Volatility Components (Test Period)', fontweight='bold')
ax4.legend(fontsize=7); ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(5, 2, 7)
ax5.plot(test_index, actual_returns, color='steelblue', linewidth=0.6, alpha=0.7)
ax5.plot(test_index, -var_buffered[0.95], color='orange', linewidth=1.0, linestyle='--', label='95% VaR')
ax5.scatter(test_index[breach_buffered[0.95]], actual_returns[breach_buffered[0.95]],
            color='orange', s=20, zorder=5, label=f"Breaches: {breach_buffered[0.95].sum()}")
ax5.set_title(f"95% VaR -- Breach Rate: {breach_buffered[0.95].mean()*100:.2f}% (Expected 5.00%)", fontweight='bold')
ax5.legend(fontsize=7); ax5.grid(True, alpha=0.3)

ax6 = fig.add_subplot(5, 2, 8)
ax6.plot(test_index, actual_returns, color='steelblue', linewidth=0.6, alpha=0.7)
ax6.plot(test_index, -var_buffered[0.99], color='red', linewidth=1.0, linestyle='--', label='99% VaR')
ax6.scatter(test_index[breach_buffered[0.99]], actual_returns[breach_buffered[0.99]],
            color='red', s=20, zorder=5, label=f"Breaches: {breach_buffered[0.99].sum()}")
ax6.set_title(f"99% VaR -- Breach Rate: {breach_buffered[0.99].mean()*100:.2f}% (Expected 1.00%)", fontweight='bold')
ax6.legend(fontsize=7); ax6.grid(True, alpha=0.3)

ax7 = fig.add_subplot(5, 2, (9, 10))
ax7.fill_between(test_index, -var_buffered[0.99], -es_buffered[0.99], alpha=0.3, color='red',    label='ES Zone 99%')
ax7.fill_between(test_index, -var_buffered[0.95], -var_buffered[0.99],alpha=0.3, color='orange', label='VaR Zone 95-99%')
ax7.plot(test_index, -var_buffered[0.95], color='orange',  linewidth=1.0, linestyle='--', label='95% VaR')
ax7.plot(test_index, -var_buffered[0.99], color='red',     linewidth=1.0, linestyle='--', label='99% VaR')
ax7.plot(test_index, -es_buffered[0.99],  color='darkred', linewidth=1.2, label='99% ES')
ax7.plot(test_index, actual_returns,      color='steelblue',linewidth=0.6, alpha=0.5, label='Actual Returns')
ax7.set_title('Complete Dynamic VaR & ES Risk Zones -- Test Period', fontweight='bold')
ax7.set_xlabel('Date'); ax7.legend(fontsize=7, ncol=3); ax7.grid(True, alpha=0.3)

plt.tight_layout()
output_file = 'ensemble_risk_report_FTSE.png'
plt.savefig(output_file, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {output_file}")

# Text summary
sep = "=" * 62
print(sep)
print(f"  ENSEMBLE GARCH+LSTM SUMMARY -- FTSE 100")
print(f"  Generated: {report_date}")
print(sep)
print(f"  GJR-GARCH  alpha={gjr_result.params['alpha[1]']:.4f}  gamma={gjr_result.params['gamma[1]']:.4f}  beta={gjr_result.params['beta[1]']:.4f}  nu={nu:.2f}  Persistence={persistence:.4f}")
print(f"  LSTM       Best Epoch={best_epoch}  Val Loss={best_val_loss:.6f}  Test MAE={test_mae:.6f}")
print(f"  Ensemble   k={best_buffer:.2f}  GARCH={GARCH_WEIGHT}  LSTM={LSTM_WEIGHT}")
print(f"  Annualised 95% VaR={annual_var_95*100:.2f}%  99% VaR={annual_var_99*100:.2f}%")
print(f"  Annualised 95% ES ={annual_es_95*100:.2f}%   99% ES ={annual_es_99*100:.2f}%")
print(f"  95% Breach Rate : {breach_buffered[0.95].mean()*100:.2f}%  (Expected 5.00%)")
print(f"  99% Breach Rate : {breach_buffered[0.99].mean()*100:.2f}%  (Expected 1.00%)")
print(sep)